# 01 · Tokens, embeddings and positions

**Transformer Models · Notebook 1 of 4**

## What you will build

A transformer never sees text. It sees a grid of numbers with one row per token. In this notebook you build the path from
one to the other: a character tokenizer whose `decode(encode(text))` gives back exactly the text it started from, a few
byte-pair-encoding merges that trade a larger vocabulary for shorter sequences, the embedding lookup that turns token ids
into trainable vectors, and the sinusoidal signals that tell the model where each token sits. At the end, a batch of
sentences becomes a `(batch, length, width)` tensor: the input to the attention you build in notebook 02.

## Learning objectives

By the end of this notebook you will be able to:

1. build a character vocabulary with a reserved padding id, and encode and decode text with an exact round trip;
2. learn byte-pair-encoding merges on a small corpus and measure how they shorten the text as the vocabulary grows;
3. look up embeddings with the tiny tensor engine, and explain why a token that appears twice receives the sum of two gradients;
4. compute sinusoidal positional encodings, and show by experiment why a model needs positions to tell "the cat likes the dog." from "the dog likes the cat.".

## Dependencies

**Tier 1: NumPy and Matplotlib.** Everything runs on NumPy and the tiny tensor engine from Base Model Training, which is
pasted into a hidden recap cell below. One plotting cell in section 4 uses Matplotlib, and nothing later depends on it.
The notebook needs no internet access. It runs the same way in JupyterLite (in your browser), on your JupyterHub server
and in Google Colab, and it takes a few seconds from top to bottom.

### How to use this notebook

Each building block follows the same rhythm: a short explanation, a tiny demo, then **✍️ Your turn**, where you type a function into a stub and run the check cell under it. The check prints ✅ when your function matches the reference and ❌, with the failing input, when it does not. Until your version passes, the notebook quietly uses the reference, so *Run all* always works; once it passes, every later cell runs on **your** code. Cells marked 🧪 are experiments: change the marked setting and run them again.

## Setup

Run this cell first. It loads NumPy and Matplotlib and defines `practice`, the checker used after every ✍️ cell.

In [ ]:
# ── Setup ─ run this cell first ────────────────────────────────────────────
import numpy as np

rng = np.random.default_rng(0)


def _close(got, want, atol):
    if isinstance(want, dict):
        return isinstance(got, dict) and got.keys() == want.keys() and all(
            _close(got[key], want[key], atol) for key in want
        )
    if isinstance(want, (list, tuple)):
        return isinstance(got, (list, tuple)) and len(got) == len(want) and all(
            _close(g, w, atol) for g, w in zip(got, want)
        )
    try:
        got_array, want_array = np.asarray(got, dtype=float), np.asarray(want, dtype=float)
    except (TypeError, ValueError):
        return got == want
    return got_array.shape == want_array.shape and np.allclose(got_array, want_array, atol=atol)


def _short(value):
    if isinstance(value, tuple):
        return "(" + ", ".join(_short(item) for item in value) + ")"
    if isinstance(value, np.ndarray):
        text = np.array2string(value, precision=4, threshold=8, separator=", ").replace("\n", "")
    else:
        text = repr(value)
    return text if len(text) < 120 else text[:117] + "..."


def practice(name, yours, reference, cases, atol=1e-6):
    """Check your function against the reference on a few inputs.

    Returns the function the rest of the notebook should use: yours once it
    matches, otherwise the reference -- so you can always keep going.
    """
    try:
        for args in cases:
            got, want = yours(*args), reference(*args)
            if not _close(got, want, atol):
                print(f"❌ {name}: for input {_short(args)}")
                print(f"   you returned {_short(got)}")
                print(f"   expected     {_short(want)}")
                print(f"   The rest of the notebook uses the reference {name} until yours matches.")
                return reference
    except NotImplementedError:
        print(f"✍️  {name}: not written yet. Type it in the cell above and run this cell again.")
        print("   Until then the rest of the notebook uses the reference version.")
        return reference
    except Exception as error:  # a typo in the learner's code must not stop the lesson
        print(f"❌ {name} raised {type(error).__name__}: {error}")
        print(f"   The rest of the notebook uses the reference {name} until yours runs.")
        return reference
    print(f"✅ {name}: matches the reference on {len(cases)} test case{'s' if len(cases) != 1 else ''}. From here on the notebook runs YOUR {name}.")
    return yours
import matplotlib.pyplot as plt

## 📦 Recap: the tiny tensor engine you built in Base Model Training

The hidden cell below is the reverse-mode autograd engine from Base Model Training, unchanged. `Tensor` supports `+ - * / ** @`, `exp`, `log`, `tanh`, `relu`, `sigmoid`, `sum`, `mean`, `reshape`, `transpose`, NumPy-style indexing, `softmax`, `log_softmax` and `backward()`; next to it are `parameter`, `zeros_parameter`, `zero_grad`, `cross_entropy`, `layer_norm`, `numerical_grad` and `relative_error`. Every operation in it has been checked against numerical gradients. Run the cell, and open it if you want to see again how an operation passes its gradient back.

In [ ]:
# ── Tiny tensor autograd (NumPy) ── the engine you built in Base Model Training ──
import numpy as np


def _unbroadcast(grad, shape):
    """Sum a gradient back down to `shape`, undoing NumPy broadcasting."""
    while grad.ndim > len(shape):
        grad = grad.sum(axis=0)
    for axis, size in enumerate(shape):
        if size == 1 and grad.shape[axis] != 1:
            grad = grad.sum(axis=axis, keepdims=True)
    return grad


class Tensor:
    """A NumPy array that remembers how it was made, so gradients can flow back."""

    def __init__(self, data, requires_grad=False, parents=(), backward=None, op=""):
        self.data = np.asarray(data, dtype=np.float64)
        self.requires_grad = requires_grad
        self.grad = None
        self._parents = parents
        self._backward = backward  # grad_out -> one gradient per parent
        self.op = op

    shape = property(lambda self: self.data.shape)
    ndim = property(lambda self: self.data.ndim)
    T = property(lambda self: self.transpose())

    def __repr__(self):
        return f"Tensor(shape={self.shape}, op={self.op or 'leaf'}, requires_grad={self.requires_grad})"

    @staticmethod
    def lift(value):
        return value if isinstance(value, Tensor) else Tensor(value)

    def _result(self, data, parents, backward, op):
        if any(parent.requires_grad for parent in parents):
            return Tensor(data, True, parents, backward, op)
        return Tensor(data, op=op)

    # arithmetic, with broadcasting
    def __add__(self, other):
        other = Tensor.lift(other)
        return self._result(self.data + other.data, (self, other), lambda g: (g, g), "add")

    __radd__ = __add__

    def __neg__(self):
        return self._result(-self.data, (self,), lambda g: (-g,), "neg")

    def __sub__(self, other):
        return self + (-Tensor.lift(other))

    def __rsub__(self, other):
        return Tensor.lift(other) + (-self)

    def __mul__(self, other):
        other = Tensor.lift(other)
        return self._result(self.data * other.data, (self, other),
                            lambda g: (g * other.data, g * self.data), "mul")

    __rmul__ = __mul__

    def __truediv__(self, other):
        other = Tensor.lift(other)
        return self._result(self.data / other.data, (self, other),
                            lambda g: (g / other.data, -g * self.data / other.data ** 2), "div")

    def __rtruediv__(self, other):
        return Tensor.lift(other) / self

    def __pow__(self, exponent):
        assert isinstance(exponent, (int, float)), "only constant exponents"
        return self._result(self.data ** exponent, (self,),
                            lambda g: (g * exponent * self.data ** (exponent - 1),), "pow")

    def __matmul__(self, other):
        other = Tensor.lift(other)

        def backward(g):
            return (g @ np.swapaxes(other.data, -1, -2), np.swapaxes(self.data, -1, -2) @ g)

        return self._result(self.data @ other.data, (self, other), backward, "matmul")

    # elementwise functions
    def exp(self):
        out = np.exp(self.data)
        return self._result(out, (self,), lambda g: (g * out,), "exp")

    def log(self):
        return self._result(np.log(self.data), (self,), lambda g: (g / self.data,), "log")

    def tanh(self):
        out = np.tanh(self.data)
        return self._result(out, (self,), lambda g: (g * (1 - out ** 2),), "tanh")

    def relu(self):
        return self._result(np.maximum(self.data, 0), (self,), lambda g: (g * (self.data > 0),), "relu")

    def sigmoid(self):
        out = 0.5 * (1 + np.tanh(0.5 * self.data))  # stable for large |x|
        return self._result(out, (self,), lambda g: (g * out * (1 - out),), "sigmoid")

    # reductions and reshaping
    def sum(self, axis=None, keepdims=False):
        def backward(g):
            if axis is not None and not keepdims:
                g = np.expand_dims(g, axis)
            return (np.broadcast_to(g, self.shape).copy(),)

        return self._result(self.data.sum(axis=axis, keepdims=keepdims), (self,), backward, "sum")

    def mean(self, axis=None, keepdims=False):
        count = self.data.size if axis is None else np.prod([self.shape[a] for a in np.atleast_1d(axis)])
        return self.sum(axis=axis, keepdims=keepdims) / count

    def reshape(self, *shape):
        return self._result(self.data.reshape(*shape), (self,), lambda g: (g.reshape(self.shape),), "reshape")

    def transpose(self, *axes):
        axes = axes or tuple(reversed(range(self.ndim)))
        inverse = np.argsort(axes)
        return self._result(self.data.transpose(axes), (self,), lambda g: (g.transpose(inverse),), "transpose")

    def __getitem__(self, index):
        def backward(g):
            full = np.zeros_like(self.data)
            np.add.at(full, index, g)  # repeated indices (an embedding row used twice) add up
            return (full,)

        return self._result(self.data[index], (self,), backward, "index")

    def log_softmax(self, axis=-1):
        shifted = self.data - self.data.max(axis=axis, keepdims=True)
        out = shifted - np.log(np.exp(shifted).sum(axis=axis, keepdims=True))
        probs = np.exp(out)
        return self._result(out, (self,), lambda g: (g - probs * g.sum(axis=axis, keepdims=True),), "log_softmax")

    def softmax(self, axis=-1):
        shifted = np.exp(self.data - self.data.max(axis=axis, keepdims=True))
        out = shifted / shifted.sum(axis=axis, keepdims=True)
        return self._result(out, (self,), lambda g: (out * (g - (g * out).sum(axis=axis, keepdims=True)),), "softmax")

    # reverse-mode differentiation
    def backward(self, grad=None):
        order, seen = [], set()

        def visit(node):
            if id(node) not in seen:
                seen.add(id(node))
                for parent in node._parents:
                    visit(parent)
                order.append(node)

        visit(self)
        self.grad = np.ones_like(self.data) if grad is None else np.asarray(grad, dtype=np.float64)
        for node in reversed(order):
            if node._backward is None or node.grad is None:
                continue
            for parent, parent_grad in zip(node._parents, node._backward(node.grad)):
                if not parent.requires_grad or parent_grad is None:
                    continue
                parent_grad = _unbroadcast(np.asarray(parent_grad, dtype=np.float64), parent.shape)
                parent.grad = parent_grad if parent.grad is None else parent.grad + parent_grad


def parameter(*shape, rng, scale=None):
    """A trainable weight, initialised with N(0, scale^2); default scale is He's sqrt(2 / fan_in)."""
    scale = np.sqrt(2.0 / shape[0]) if scale is None else scale
    return Tensor(rng.normal(0.0, scale, shape), requires_grad=True)


def zeros_parameter(*shape):
    return Tensor(np.zeros(shape), requires_grad=True)


def zero_grad(params):
    for p in params:
        p.grad = None


def cross_entropy(logits, targets, mask=None):
    """Mean next-token/class cross-entropy. logits: (..., classes); targets: int (...); mask: 1.0 keeps a position."""
    flat = logits.reshape(-1, logits.shape[-1]).log_softmax(axis=-1)
    targets = np.asarray(targets).reshape(-1)
    picked = flat[np.arange(targets.size), targets]
    if mask is None:
        return -picked.mean()
    weights = np.asarray(mask, dtype=np.float64).reshape(-1)
    return -(picked * weights).sum() / max(weights.sum(), 1.0)


def layer_norm(x, gamma, beta, eps=1e-5):
    mean = x.mean(axis=-1, keepdims=True)
    centred = x - mean
    var = (centred ** 2).mean(axis=-1, keepdims=True)
    return centred / (var + eps) ** 0.5 * gamma + beta


def numerical_grad(f, tensor, eps=1e-6):
    """Central-difference gradient of scalar f() with respect to tensor.data."""
    grad = np.zeros_like(tensor.data)
    it = np.nditer(tensor.data, flags=["multi_index"])
    for _ in it:
        index = it.multi_index
        original = tensor.data[index]
        tensor.data[index] = original + eps
        plus = float(f().data)
        tensor.data[index] = original - eps
        minus = float(f().data)
        tensor.data[index] = original
        grad[index] = (plus - minus) / (2 * eps)
    return grad


def relative_error(a, b):
    return float(np.linalg.norm(a - b) / max(np.linalg.norm(a) + np.linalg.norm(b), 1e-12))

### One more checker: `practice_tensor`

`practice` compares plain values, but most functions in this course take and return `Tensor`s. `practice_tensor` (hidden below) runs your function and the reference on the same inputs and compares two things: the values they return, and the gradients that flow back to every input that requires one. A function that slips into raw NumPy (through `.data`, `np.exp` or `float()`, for example) can return the right numbers while cutting the gradient path, so training would silently stop updating some weights. The second comparison catches that.

In [ ]:
# ── practice_tensor: practice() for functions built from Tensor operations ──


def _tensors_in(value):
    """Every Tensor inside value, looking into tuples, lists and dicts."""
    if isinstance(value, Tensor):
        return [value]
    if isinstance(value, dict):
        value = list(value.values())
    if isinstance(value, (list, tuple)):
        return [tensor for item in value for tensor in _tensors_in(item)]
    return []


def _values_and_gradients(function, args):
    """Run function(*args). Return its output arrays and the gradients it sends back to each input Tensor."""
    inputs = [tensor for tensor in _tensors_in(args) if tensor.requires_grad]
    zero_grad(inputs)
    result = function(*args)
    outputs = list(result) if isinstance(result, tuple) else [result]
    for output in outputs:
        if not isinstance(output, Tensor):
            hint = ("; a NumPy array on the left of + or @ turns the result into an object array, so put the Tensor first"
                    if getattr(output, "dtype", None) == object else "; build the result from Tensor operations")
            raise TypeError(f"expected a Tensor but got {type(output).__name__}{hint}")
    probe = Tensor(0.0)
    for output in outputs:  # an uneven weighting, so that no gradient cancels out by symmetry
        probe = probe + (output * np.cos(np.arange(output.data.size)).reshape(output.shape)).sum()
    probe.backward()
    values = [output.data.copy() for output in outputs]
    gradients = [np.zeros_like(tensor.data) if tensor.grad is None else tensor.grad.copy() for tensor in inputs]
    zero_grad(inputs)
    return values, gradients


def practice_tensor(name, yours, reference, cases, atol=1e-6):
    """Like practice(), for functions built from Tensor operations.

    It checks the values your function returns AND the gradients that flow back
    through them, so code that slips into raw NumPy (which cuts the gradient
    path) does not pass. Returns yours once it matches, otherwise the reference.
    """
    try:
        for args in cases:
            got, got_gradients = _values_and_gradients(yours, args)
            want, want_gradients = _values_and_gradients(reference, args)
            if not _close(got, want, atol):
                print(f"❌ {name}: for input {_short(args)}")
                print(f"   you returned {_short(got[0] if len(got) == 1 else tuple(got))}")
                print(f"   expected     {_short(want[0] if len(want) == 1 else tuple(want))}")
                print(f"   The rest of the notebook uses the reference {name} until yours matches.")
                return reference
            if not _close(got_gradients, want_gradients, atol):
                print(f"❌ {name}: the values match, but the gradients flowing back through your function do not.")
                print("   Use Tensor operations only: .data, np.* functions and float() cut the gradient path.")
                print(f"   The rest of the notebook uses the reference {name} until yours matches.")
                return reference
    except NotImplementedError:
        print(f"✍️  {name}: not written yet. Type it in the cell above and run this cell again.")
        print("   Until then the rest of the notebook uses the reference version.")
        return reference
    except Exception as error:  # a typo in the learner's code must not stop the lesson
        print(f"❌ {name} raised {type(error).__name__}: {error}")
        print(f"   The rest of the notebook uses the reference {name} until yours runs.")
        return reference
    print(f"✅ {name}: values and gradients match the reference on {len(cases)} test case{'s' if len(cases) != 1 else ''}. "
          f"From here on the notebook runs YOUR {name}.")
    return yours

## 1 · Characters become ids

A model computes with numbers, so text must first become a sequence of integers, its **token ids**. The simplest
tokenizer uses one token per character. Its vocabulary is the sorted list of characters in the training text, and a
character's id is its position in that list. Two lookups go with it: `itos` ("integer to string") maps an id to its
text, and `stoi` maps text back to its id.

We reserve id 0 for a special token, `<pad>`. Sentences have different lengths, and to stack them into one array we fill
the short ones with padding. Keeping padding out of the real vocabulary means a padded position can never be mistaken
for a character.

**Shapes.** A string of $n$ characters becomes a list of $n$ ids. A batch of $B$ sentences, padded to the longest length
$L$, becomes an integer array of shape $(B, L)$.

The course uses one small generated grammar throughout. Its sentences follow a rule you will meet again in notebook 04:
the verb agrees with its subject ("the cat likes", "the cats like").

In [ ]:
NOUNS = ["cat", "dog", "bird", "frog", "hen"]
VERBS = ["like", "help", "find", "call"]


def random_sentence(rng):
    """One sentence of the course grammar: the verb agrees with its subject."""
    subject, obj = rng.choice(NOUNS, size=2)
    verb = rng.choice(VERBS)
    plural, object_plural = rng.integers(0, 2, size=2)
    return f"the {subject}{'s' * plural} {verb}{'s' * (1 - plural)} the {obj}{'s' * object_plural}."


corpus = [random_sentence(np.random.default_rng(seed)) for seed in range(40)]
itos = ["<pad>"] + sorted(set("".join(corpus)))
stoi = {token: index for index, token in enumerate(itos)}
print(corpus[:3])
print(f"{len(itos)} tokens:", itos)
print("'c' has id", stoi["c"], "and id", stoi["c"], "means", repr(itos[stoi["c"]]))

### ✍️ Your turn: `encode`

Write `encode(text, stoi)`. It returns the list of ids for the characters of `text`, in order; `stoi` is a dict from
character to id. An empty string gives an empty list.

*Hint:* a list comprehension over the characters of `text` does it in one line.

Type it rather than pasting it: typing is how the shapes and indices sink in. The reference is in the collapsed cell below; open it only if you are stuck.

In [ ]:
def encode_reference(text, stoi):
    """Return the list of token ids for the characters of text, in order."""
    return [stoi[character] for character in text]

In [ ]:
def encode(text, stoi):
    """Return the list of token ids for the characters of text, in order."""
    # ✍️ Your code here (one line)
    raise NotImplementedError

In [ ]:
encode = practice("encode", encode, encode_reference, cases=[
    ("the cat", stoi),
    ("dogs.", stoi),
    ("", stoi),
])

### ✍️ Your turn: `decode`

Write `decode(ids, itos)`. It returns the string spelled by a sequence of ids: look up each id in the list `itos` and
join the pieces.

*Hint:* `"".join(...)` glues a sequence of strings together.

Type it rather than pasting it: typing is how the shapes and indices sink in. The reference is in the collapsed cell below; open it only if you are stuck.

In [ ]:
def decode_reference(ids, itos):
    """Return the string spelled by a sequence of token ids."""
    return "".join(itos[index] for index in ids)

In [ ]:
def decode(ids, itos):
    """Return the string spelled by a sequence of token ids."""
    # ✍️ Your code here (one line)
    raise NotImplementedError

In [ ]:
decode = practice("decode", decode, decode_reference, cases=[
    (encode_reference("the hen.", stoi), itos),
    ([], itos),
    ([0, 0], itos),
])

**Expected behaviour.** `encode` and `decode` undo each other, so `decode(encode(s))` gives back `s` exactly; the next
cell checks this for every sentence in the corpus. Notice the last test case: padding decodes to the literal text
`<pad>`, so strip padding before you print a padded row.

In [ ]:
for sentence in corpus[:3]:
    ids = encode(sentence, stoi)
    print(ids[:10], "...", "round trip exact:", decode(ids, itos) == sentence)
assert all(decode(encode(sentence, stoi), itos) == sentence for sentence in corpus), "decode(encode(s)) must give back s"

## 2 · Byte-pair encoding: merge the most frequent pair

Character tokens keep the vocabulary tiny but make sequences long, and the cost of attention grows with the *square* of
the sequence length (notebook 04 measures this). Subword tokenizers sit in between. **Byte-pair encoding** (BPE), which
Sennrich, Haddow and Birch (2016) adapted for neural machine translation, starts from characters (production tokenizers
start from bytes) and repeats one step:

1. count every adjacent pair of tokens in the training text;
2. replace the most frequent pair everywhere by a new token with the next unused id.

Each merge adds one vocabulary entry and shortens every sequence that contains the pair. The ordered list of merges *is*
the tokenizer: to encode new text you replay the merges in the order they were learned, so the list has to be saved
with the model.

**Shapes.** `ids` is a list of $n$ ints, a pair is a tuple `(a, b)`, and one merge returns a list of at most $n$ ints.

In [ ]:
ids = encode("the cat likes the dog.", stoi)
pairs = list(zip(ids, ids[1:]))
print(len(ids), "ids give", len(pairs), "adjacent pairs; the first four:", pairs[:4])
print("as text:", [decode(pair, itos) for pair in pairs[:4]])

### ✍️ Your turn: `most_frequent_pair`

Write `most_frequent_pair(ids)`. It returns the adjacent pair `(ids[i], ids[i + 1])` that occurs most often in `ids`,
as a tuple. Count overlapping occurrences too: `[5, 5, 5]` contains the pair `(5, 5)` twice. If several pairs share
the highest count, return the one whose **first occurrence comes earliest** in `ids`. You may assume that `ids` has at
least two elements.

*Hints:* `zip(ids, ids[1:])` walks the pairs in order. A dict of counts filled in that order remembers which pair came
first, and `max(counts, key=counts.get)` returns the first of several equal maxima.

Type it rather than pasting it: typing is how the shapes and indices sink in. The reference is in the collapsed cell below; open it only if you are stuck.

In [ ]:
def most_frequent_pair_reference(ids):
    """The most frequent adjacent pair in ids; ties go to the pair that occurs first."""
    counts = {}
    for pair in zip(ids, ids[1:]):
        counts[pair] = counts.get(pair, 0) + 1
    return max(counts, key=counts.get)

In [ ]:
def most_frequent_pair(ids):
    """The most frequent adjacent pair in ids; ties go to the pair that occurs first."""
    # ✍️ Your code here (about four lines)
    raise NotImplementedError

In [ ]:
most_frequent_pair = practice("most_frequent_pair", most_frequent_pair, most_frequent_pair_reference, cases=[
    ([1, 2, 3, 1, 2],),
    ([1, 2, 3, 1, 2, 3],),  # (1, 2) and (2, 3) both occur twice: (1, 2) comes first
    ([5, 5, 5, 7],),        # overlapping pairs count
    (encode_reference("the cat likes the dog.", stoi),),
])

### ✍️ Your turn: `merge_pair`

Write `merge_pair(ids, pair, new_id)`. It returns a **new** list in which every occurrence of `pair`, scanning from left
to right, is replaced by `new_id`. Once two ids have been merged, neither can take part in another merge in the same
pass, so `merge_pair([5, 5, 5], (5, 5), 9)` is `[9, 5]`. Do not change `ids` itself.

*Hint:* walk an index `i` with a `while` loop. When `i + 1 < len(ids)` and `(ids[i], ids[i + 1]) == pair`, append
`new_id` and move two places; otherwise append `ids[i]` and move one.

Type it rather than pasting it: typing is how the shapes and indices sink in. The reference is in the collapsed cell below; open it only if you are stuck.

In [ ]:
def merge_pair_reference(ids, pair, new_id):
    """A new list in which every occurrence of pair, scanning left to right, becomes new_id."""
    merged, i = [], 0
    while i < len(ids):
        if i + 1 < len(ids) and (ids[i], ids[i + 1]) == pair:
            merged.append(new_id)
            i += 2
        else:
            merged.append(ids[i])
            i += 1
    return merged

In [ ]:
def merge_pair(ids, pair, new_id):
    """A new list in which every occurrence of pair, scanning left to right, becomes new_id."""
    # ✍️ Your code here (about eight lines)
    raise NotImplementedError

In [ ]:
merge_pair = practice("merge_pair", merge_pair, merge_pair_reference, cases=[
    ([1, 2, 3, 1, 2], (1, 2), 9),
    ([5, 5, 5], (5, 5), 9),
    ([1, 2], (2, 1), 9),
    ([7], (7, 7), 9),
])

**Expected behaviour.** `merge_pair([1, 2, 3, 1, 2], (1, 2), 9)` gives `[9, 3, 9]`, and a list without the pair comes
back unchanged. The function below, `learn_merges`, repeats your two functions to learn a tokenizer: each merge gets the
next free id, and the new token's text is the two old texts glued together.

In [ ]:
def learn_merges(ids, num_merges, tokens):
    """Learn num_merges byte-pair merges on ids. tokens maps id -> text and grows by one entry per merge."""
    tokens = list(tokens)
    merges = []
    for _ in range(num_merges):
        if len(ids) < 2:
            break
        pair = most_frequent_pair(ids)
        new_id = len(tokens)
        ids = merge_pair(ids, pair, new_id)
        merges.append((pair, new_id))
        tokens.append(tokens[pair[0]] + tokens[pair[1]])
    return ids, merges, tokens


corpus_text = " ".join(corpus)
corpus_ids = encode(corpus_text, stoi)
num_merges = 10  # 🧪 try 0, 30 and 60, then run this cell again
merged_ids, merges, tokens = learn_merges(corpus_ids, num_merges, itos)
print(f"{num_merges} merges: vocabulary {len(itos)} -> {len(tokens)} tokens; corpus {len(corpus_ids)} -> {len(merged_ids)} tokens")
print("new tokens, in the order learned:", [tokens[new_id] for _, new_id in merges])
print("the corpus now starts:", [tokens[index] for index in merged_ids[:10]])
assert decode(merged_ids, tokens) == corpus_text, "merging must never change the text"

**What to notice.** The first merges join the most common neighbours: `"he"`, then `"the"`, then `"the "`. Some merges
cross a sentence boundary, such as `". the "`, because we joined the sentences with spaces before learning. Production
tokenizers first split text into words or chunks so that merges never cross those boundaries. The decoded text is
unchanged by every merge: BPE changes how the text is cut up, never what it says.

## 3 · Embedding lookup: ids become vectors

An **embedding table** $E \in \mathbb{R}^{V \times D}$ holds one learned row of width $D$ for each of the $V$ tokens.
Looking up ids of shape $(B, L)$ gives vectors of shape $(B, L, D)$:

$$x_{b,i} = E_{\,\mathrm{ids}_{b,i}}.$$

The lookup equals multiplying a one-hot matrix by $E$, but indexing skips all the multiplications by zero. The id
itself carries no numeric meaning (token 8 is not "twice" token 4); whatever the rows come to mean is learned from the
training objective.

The gradient follows the same rows back. The gradient for row $v$ is the sum of the gradients of every position that
holds token $v$:

$$\frac{\partial \mathcal{L}}{\partial E_v} = \sum_{(b,i)\,:\,\mathrm{ids}_{b,i} = v} \frac{\partial \mathcal{L}}{\partial x_{b,i}},$$

so rows of tokens that did not appear get zero, and a token used twice gets the sum of two contributions. The engine's
indexing operation already does this with `np.add.at`.

In [ ]:
table_demo = Tensor(np.arange(12.0).reshape(4, 3))  # 4 tokens, width 3
ids_demo = np.array([2, 0, 2])
one_hot = np.eye(4)[ids_demo]  # (3, 4): one row per position, with a single 1
print(one_hot @ table_demo.data)  # the matrix product just picks rows 2, 0 and 2 of the table

### ✍️ Your turn: `embedding_lookup`

Write `embedding_lookup(table, ids)`. `table` is a `Tensor` of shape `(vocab, width)` and `ids` is an integer NumPy
array of any shape, for example `(batch, length)`. Return a `Tensor` of shape `ids.shape + (width,)` holding the rows
of `table` for each id. Gradients must flow back into `table`.

*Hint:* the engine's `Tensor` supports NumPy indexing, and its backward pass already adds up repeated rows. One line.

Type it rather than pasting it: typing is how the shapes and indices sink in. The reference is in the collapsed cell below; open it only if you are stuck.

In [ ]:
def embedding_lookup_reference(table, ids):
    """Rows of the embedding table for each id. table: Tensor (vocab, width); ids: ints of any shape."""
    return table[ids]

In [ ]:
def embedding_lookup(table, ids):
    """Rows of the embedding table for each id. table: Tensor (vocab, width); ids: ints of any shape."""
    # ✍️ Your code here (one line)
    raise NotImplementedError

In [ ]:
table_check = parameter(5, 3, rng=np.random.default_rng(1), scale=1.0)
embedding_lookup = practice_tensor("embedding_lookup", embedding_lookup, embedding_lookup_reference, cases=[
    (table_check, np.array([[0, 2, 2], [4, 1, 0]])),  # token 2 appears twice in the first row
    (table_check, np.array([3])),
    (table_check, np.array([[1]])),
])

**Expected behaviour.** Shapes follow the ids: `(2, 3)` ids give a `(2, 3, 3)` tensor. In the next cell every gradient
of the output is 1, so row $v$ of `table.grad` holds the number of times token $v$ appears: tokens 1 and 3 never appear,
and their rows stay at zero.

In [ ]:
table = parameter(5, 3, rng=np.random.default_rng(1), scale=1.0)
ids_twice = np.array([[0, 2, 2], [4, 2, 0]])
embedding_lookup(table, ids_twice).sum().backward()
print(table.grad)
counts = np.bincount(ids_twice.ravel(), minlength=5)
assert np.allclose(table.grad, counts[:, None] * np.ones((1, 3))), "row v should hold the count of token v"
print("parameters in a", len(itos), "x 32 table:", len(itos) * 32)

## 4 · Positions: why order needs a signal

In notebook 02 you build **attention**, which mixes the vectors of a sequence with weights computed from the vectors
themselves. Nothing in that recipe looks at *where* a vector sits. Shuffle the tokens and attention shuffles its outputs
in the same way; average them and the result does not change at all. "the cat likes the dog." and "the dog likes the
cat." contain exactly the same characters, so an order-blind model sees the same bag of vectors.

The demo checks this with a weight-free stand-in for attention, $\mathrm{mix}(X) = \mathrm{softmax}(XX^\top/\sqrt{D})\,X$,
applied to both sentences and averaged over positions.

In [ ]:
def mix(x):
    """A weight-free stand-in for attention: softmax(x x^T / sqrt(D)) x, for x of shape (length, width)."""
    return (x @ x.T / np.sqrt(x.shape[-1])).softmax(axis=-1) @ x


E_demo = parameter(len(itos), 16, rng=np.random.default_rng(3), scale=1.0)
cat_first = np.array(encode("the cat likes the dog.", stoi))
dog_first = np.array(encode("the dog likes the cat.", stoi))
summary_cat = mix(embedding_lookup(E_demo, cat_first)).mean(axis=0).data
summary_dog = mix(embedding_lookup(E_demo, dog_first)).mean(axis=0).data
print("largest difference between the two sentence summaries:", np.abs(summary_cat - summary_dog).max())

The difference is zero up to rounding: without positions, the model cannot tell who likes whom.

### Sinusoidal positional encodings

Vaswani et al. (2017) add a fixed signal to each position's embedding. For position $p$ and column pair
$i = 0, 1, \dots, D/2 - 1$:

$$PE_{p,\,2i} = \sin\!\left(\frac{p}{10000^{2i/D}}\right), \qquad PE_{p,\,2i+1} = \cos\!\left(\frac{p}{10000^{2i/D}}\right).$$

Each column pair is a clock hand turning at its own rate: the first pair completes a turn every $2\pi \approx 6.3$
positions, while the last barely moves over thousands. Together they give every position a distinct pattern, and
because a shift of $k$ positions rotates each pair by a fixed angle, the same weights can recognise "three positions
back" anywhere in a sequence. The matrix has shape $(L, D)$, is added to the $(B, L, D)$ embeddings by broadcasting and
has no trainable parameters. Learned position tables and rotary embeddings are common alternatives; the lesson notes
compare them.

In [ ]:
rates = 1.0 / 10000 ** (np.arange(0, 8, 2) / 8)  # one rate per column pair, for width D = 8
print("rates for D = 8:", rates)
print("angles at position 3:", 3 * rates)

### ✍️ Your turn: `sinusoidal_positions`

Write `sinusoidal_positions(length, width)`. It returns a NumPy array of shape `(length, width)` whose even columns
`0, 2, 4, ...` hold $\sin(p \cdot r_i)$ and whose odd columns `1, 3, 5, ...` hold $\cos(p \cdot r_i)$, where
$r_i = 1 / 10000^{2i/D}$ is the rate of column pair $i$ (as in the demo). `width` is even.

*Hints:* `np.arange(length)[:, None]` has shape `(length, 1)` and the rates have shape `(width // 2,)`, so their product
broadcasts to `(length, width // 2)`. Start from `np.zeros((length, width))` and fill `signals[:, 0::2]` and
`signals[:, 1::2]`.

Type it rather than pasting it: typing is how the shapes and indices sink in. The reference is in the collapsed cell below; open it only if you are stuck.

In [ ]:
def sinusoidal_positions_reference(length, width):
    """Sinusoidal position signals, shape (length, width); width must be even."""
    positions = np.arange(length)[:, None]
    rates = 1.0 / 10000 ** (np.arange(0, width, 2) / width)
    signals = np.zeros((length, width))
    signals[:, 0::2] = np.sin(positions * rates)
    signals[:, 1::2] = np.cos(positions * rates)
    return signals

In [ ]:
def sinusoidal_positions(length, width):
    """Sinusoidal position signals, shape (length, width); width must be even."""
    # ✍️ Your code here (about five lines)
    raise NotImplementedError

In [ ]:
sinusoidal_positions = practice("sinusoidal_positions", sinusoidal_positions, sinusoidal_positions_reference, cases=[
    (4, 6),
    (1, 4),   # a single position: [0, 1, 0, 1]
    (10, 8),
])

**Expected behaviour.** Row 0 is `[0, 1, 0, 1, ...]`, because $\sin 0 = 0$ and $\cos 0 = 1$. Every value lies in
$[-1, 1]$. In the plot, the left columns oscillate quickly and the right columns slowly, and the dot product between
positions falls off as they move apart: nearby positions look alike.

In [ ]:
signals = sinusoidal_positions(48, 32)
fig, axes = plt.subplots(1, 2, figsize=(10, 3.2))
axes[0].imshow(signals.T, aspect="auto", cmap="RdBu", vmin=-1, vmax=1)
axes[0].set_xlabel("position"); axes[0].set_ylabel("column"); axes[0].set_title("sinusoidal_positions(48, 32)")
axes[1].plot(signals @ signals[20], marker=".")
axes[1].set_xlabel("position"); axes[1].set_ylabel("dot product with position 20")
axes[1].set_title("nearby positions look alike")
plt.tight_layout()
plt.show()

## 5 · 🧪 Experiments

### Positions break the symmetry

Add your positional signals to the embeddings and repeat the order test from section 4. Then set `use_positions = False`
and run the cell again.

In [ ]:
use_positions = True  # 🧪 set to False and run this cell again
P = sinusoidal_positions(len(cat_first), 16) if use_positions else 0.0
summary_cat = mix(embedding_lookup(E_demo, cat_first) + P).mean(axis=0).data
summary_dog = mix(embedding_lookup(E_demo, dog_first) + P).mean(axis=0).data
print("positions used:", use_positions, "| largest difference between the summaries:", np.abs(summary_cat - summary_dog).max())

### Vocabulary size against sequence length

More merges mean a bigger vocabulary (a bigger embedding table and output layer) and shorter sequences (less attention
work per sentence). The table learns 0 to 80 merges on the same corpus.

In [ ]:
merge_counts = (0, 5, 10, 20, 40, 80)  # 🧪 add 120 and run again
print("merges  vocabulary  corpus tokens  characters per token")
for n in merge_counts:
    merged, _, grown = learn_merges(corpus_ids, n, itos)
    print(f"{n:6d}  {len(grown):10d}  {len(merged):13d}  {len(corpus_ids) / len(merged):20.2f}")

**What to notice.** The first merges pay off most, because they capture "the " and the other very frequent pieces; later
merges each save less. On this tiny, repetitive corpus the text shrinks severalfold. Natural text compresses less, and a
tokenizer learned on one kind of text cuts other kinds (another language, or code) into many more tokens.

## 6 · Put it together: from sentences to a `(batch, length, width)` tensor

`embed_batch` chains everything you wrote: encode each sentence, pad to the longest with id 0, look up the embeddings
and add the positional signals. The asserts check the behaviour, not just the shapes.

In [ ]:
def embed_batch(sentences, table, stoi):
    """Encode, pad with id 0 to the longest sentence, look up embeddings and add positions.
    Returns (ids: int array (B, L), x: Tensor (B, L, D))."""
    encoded = [encode(sentence, stoi) for sentence in sentences]
    length = max(len(ids) for ids in encoded)
    ids = np.zeros((len(sentences), length), dtype=int)
    for row, sequence in enumerate(encoded):
        ids[row, :len(sequence)] = sequence
    x = embedding_lookup(table, ids) + sinusoidal_positions(length, table.shape[1])
    return ids, x


width = 16
table = parameter(len(itos), width, rng=np.random.default_rng(7), scale=1.0)
batch = ["the cat likes the dog.", "the birds find the frogs.", "the hen calls the cats."]
ids, x = embed_batch(batch, table, stoi)
print("ids:", ids.shape, "| embeddings:", x.shape, "| parameters in the table:", table.data.size)
for row in ids:
    print(repr(decode([index for index in row if index != 0], itos)), "+", int((row == 0).sum()), "pad tokens")

In [ ]:
# 1. The tokenizer is lossless on the whole corpus.
assert all(decode(encode(sentence, stoi), itos) == sentence for sentence in corpus)

# 2. Padding fills the end of the shorter rows and nothing else.
lengths = [len(sentence) for sentence in batch]
assert ids.shape == (len(batch), max(lengths))
assert all((row[:n] != 0).all() and (row[n:] == 0).all() for row, n in zip(ids, lengths))

# 3. The batch has the right shape, and gradients reach exactly the rows that were used.
assert x.shape == (len(batch), max(lengths), width)
x.sum().backward()
used = np.zeros(len(itos), dtype=bool)
used[np.unique(ids)] = True  # this includes <pad>: we summed over padded positions too
assert np.all(table.grad[~used] == 0) and np.all(np.abs(table.grad[used]).sum(axis=1) > 0)

# 4. Twenty merges shorten the corpus to well under 60 % of its characters without changing the text.
merged_ids, merges, tokens = learn_merges(corpus_ids, 20, itos)
assert decode(merged_ids, tokens) == corpus_text
assert len(merged_ids) < 0.6 * len(corpus_ids), f"expected strong compression, got {len(merged_ids)} of {len(corpus_ids)}"

# 5. Without positions the two orders are indistinguishable; with positions they differ.
def summary(ids, positions):
    return mix(embedding_lookup(E_demo, ids) + positions).mean(axis=0).data

no_positions = np.abs(summary(cat_first, 0.0) - summary(dog_first, 0.0)).max()
with_positions = np.abs(summary(cat_first, sinusoidal_positions(len(cat_first), 16))
                        - summary(dog_first, sinusoidal_positions(len(dog_first), 16))).max()
assert no_positions < 1e-12 and with_positions > 1e-3, (no_positions, with_positions)

print(f"✅ Token path verified: {len(corpus)} exact round trips, a {ids.shape} id batch becomes a {x.shape} tensor,")
print(f"   20 merges shrink the corpus from {len(corpus_ids)} to {len(merged_ids)} tokens, and positions separate the two")
print(f"   word orders (difference {with_positions:.3f} with positions, {no_positions:.1e} without).")

## Exercises

These go beyond the type-alongs. The first two have checks; the last two are questions to answer in a markdown cell.

### ✍️ Your turn: `encode_with_unknown`

**Exercise 1.** `encode` raises `KeyError` for a character it has never seen. Write `encode_with_unknown(text, stoi,
unk_id)`, which maps such characters to `unk_id` instead (real tokenizers keep a special `<unk>` token for this).

*Hint:* `dict.get(key, default)`.

Type it rather than pasting it: typing is how the shapes and indices sink in. The reference is in the collapsed cell below; open it only if you are stuck.

In [ ]:
def encode_with_unknown_reference(text, stoi, unk_id):
    """Like encode, but characters missing from stoi become unk_id."""
    return [stoi.get(character, unk_id) for character in text]

In [ ]:
def encode_with_unknown(text, stoi, unk_id):
    """Like encode, but characters missing from stoi become unk_id."""
    # ✍️ Your code here (one line)
    raise NotImplementedError

In [ ]:
unk_id = len(itos)  # as if "<unk>" were appended to the vocabulary
encode_with_unknown = practice("encode_with_unknown", encode_with_unknown, encode_with_unknown_reference, cases=[
    ("the zebra", stoi, unk_id),
    ("cat", stoi, unk_id),
    ("", stoi, unk_id),
])

### ✍️ Your turn: `apply_merges`

**Exercise 2.** Write `apply_merges(ids, merges)`, which encodes new text with a learned tokenizer: replay each
`(pair, new_id)` in `merges` with `merge_pair`, in the order the merges were learned.

Type it rather than pasting it: typing is how the shapes and indices sink in. The reference is in the collapsed cell below; open it only if you are stuck.

In [ ]:
def apply_merges_reference(ids, merges):
    """Replay learned merges, in the order they were learned, on a new list of ids."""
    for pair, new_id in merges:
        ids = merge_pair(ids, pair, new_id)
    return ids

In [ ]:
def apply_merges(ids, merges):
    """Replay learned merges, in the order they were learned, on a new list of ids."""
    # ✍️ Your code here (about three lines)
    raise NotImplementedError

In [ ]:
apply_merges = practice("apply_merges", apply_merges, apply_merges_reference, cases=[
    (encode_reference("the hens call the frog.", stoi), merges),
    (encode_reference("dog", stoi), merges),
    (encode_reference("the cat", stoi), []),
])
print([tokens[index] for index in apply_merges(encode("the hens call the frog.", stoi), merges)])

**Exercise 3 (question).** A learned position table for a context of 512 positions at width 64 has how many
parameters? What happens when such a model meets position 600? Sinusoidal signals exist at every position, yet a model
trained only on short sequences can still fail on longer ones. Why?

**Exercise 4 (question).** Learn 40 merges, then encode the familiar word `"birds"` and the unseen word `"zebra"` with
`encode_with_unknown` (use `-1` as the unknown id, which no merge uses) followed by `apply_merges`. How many tokens does
each word cost? What does this suggest about tokenizers trained on one language and used on another?

## Common errors and debugging

- **`KeyError: 'z'` from `encode`.** The character is not in the vocabulary. Build the vocabulary from all the training
  text, or map unknown characters to an `<unk>` id (exercise 1).
- **`IndexError: arrays used as indices must be of integer type`.** The ids array holds floats. Create it with
  `dtype=int`.
- **A decoded row ends in `<pad><pad>`.** Strip ids equal to 0 before decoding a padded row.
- **No id reserved for padding.** A real character then shares id 0 with padding, and padded positions look like text.
- **Merges replayed in a different order.** The tokens no longer match the ones the model was trained on. Save the
  merge list, in order, with the model.
- **An odd width in `sinusoidal_positions`.** The sine and cosine columns no longer pair up. Keep `width` even.
- **`positions + x` instead of `x + positions`.** With a NumPy array on the left of `+`, NumPy builds an object array
  of little `Tensor`s instead of one `Tensor`. Put the `Tensor` first.

## Summary

- A tokenizer is a vocabulary plus rules. Character tokens give an exact round trip with a tiny vocabulary; id 0 is
  reserved for padding.
- BPE repeatedly merges the most frequent adjacent pair. More merges give a bigger vocabulary and shorter sequences, and
  the ordered merge list is part of the model.
- An embedding table maps ids to learned rows; a lookup equals a one-hot product, and repeated tokens add up their
  gradients.
- Attention by itself ignores order. Sinusoidal signals add a distinct, parameter-free pattern to every position.

## Next

Notebook 02 builds scaled dot-product attention on top of these embeddings, with the causal mask that stops a position
reading the future and the padding mask that hides the `<pad>` tokens you added here.